# S05a — Visualisation en code : Plotly, Altair et la carte des zones

En S04, nos graphiques étaient des images fixes. Aujourd'hui, ils deviennent **interactifs** :
survol, zoom, filtres, liens entre vues. Ce notebook prépare aussi la carte utilisée par l'application Streamlit.

In [1]:
pip install -q plotly geopandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys; sys.path.append("/home/jovyan/work")
from commun.donnees import lire
import pandas as pd
import plotly.express as px

import plotly.io as pio
pio.renderers.default = "iframe"

horaires = lire("gold_trajets_horaires")
zones = lire("gold_zones")
horaires["heure_pickup"] = pd.to_datetime(horaires["heure_pickup"])
horaires["heure"] = horaires["heure_pickup"].dt.hour
jours = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
horaires["jour_semaine"] = horaires["heure_pickup"].dt.dayofweek.map(dict(enumerate(jours)))

## 1. Plotly Express : une ligne par graphique

`px` prend un DataFrame et les noms de colonnes. Survolez, zoomez, cliquez sur la légende pour masquer une série.

In [3]:
par_heure = (horaires[~horaires["arrondissement"].isin(["Unknown", "N/A"])]
             .groupby(["heure", "arrondissement"], as_index=False)["courses"].sum())
px.line(par_heure, x="heure", y="courses", color="arrondissement", markers=True,
        labels={"heure": "Heure de prise en charge", "courses": "Courses sur le mois"},
        title=f"Pic de demande à {par_heure.groupby('heure')['courses'].sum().idxmax()} h")

In [4]:
# Une carte de chaleur jour × heure
matrice = horaires.groupby(["jour_semaine", "heure"], as_index=False)["courses"].sum()
ordre = jours
px.density_heatmap(matrice, x="heure", y="jour_semaine", z="courses", histfunc="sum",
                   category_orders={"jour_semaine": ordre}, color_continuous_scale="Blues",
                   labels={"heure": "Heure", "jour_semaine": "", "courses": "Courses"},
                   title="Quand les taxis sont-ils le plus demandés ?")

## 2. Altair : la grammaire graphique

Avec Altair, on décrit un graphique comme une combinaison de **marques** (points, barres, lignes)
et d'**encodages** (x, y, couleur, taille). Les vues peuvent être **liées** : une sélection dans l'une filtre l'autre.

In [5]:
import altair as alt
alt.data_transformers.disable_max_rows()

z = zones[~zones["arrondissement"].isin(["Unknown", "N/A"])].dropna(subset=["arrondissement"])
selection = alt.selection_point(fields=["arrondissement"], bind="legend")

nuage = (alt.Chart(z).mark_circle(size=60).encode(
            x=alt.X("courses:Q", scale=alt.Scale(type="log"), title="Courses sur le mois (échelle log)"),
            y=alt.Y("prix_moyen:Q", title="Prix moyen ($)"),
            color=alt.Color("arrondissement:N", title="Arrondissement"),
            opacity=alt.condition(selection, alt.value(0.9), alt.value(0.1)),
            tooltip=["zone", "arrondissement", "courses", "prix_moyen"])
         .add_params(selection)
         .properties(width=600, height=380,
                     title="Fréquentation et prix moyen des zones (cliquez sur la légende)"))
nuage

alt.Chart(...)

In [6]:
autres = (horaires[~horaires["arrondissement"].isin(["Unknown", "N/A", "Manhattan", "Queens"])]
          .groupby(["heure", "arrondissement"], as_index=False)["courses"].sum())
px.line(autres, x="heure", y="courses", color="arrondissement", markers=True,
        labels={"heure": "Heure de prise en charge", "courses": "Courses sur le mois"},
        title=f"Hors Manhattan : pic de demande à {autres.groupby('heure')['courses'].sum().idxmax()} h")

## 3. Préparer la carte des zones

Les contours des zones sont fournis au format *shapefile*, dans un système de coordonnées propre à New York.
On les convertit en **GeoJSON** en latitude/longitude (EPSG:4326), le format attendu par les cartes web.

In [7]:
import zipfile, geopandas as gpd

RAW = "/home/jovyan/work/data/raw"
with zipfile.ZipFile(f"{RAW}/taxi_zones.zip") as zf:
    zf.extractall(f"{RAW}/taxi_zones")

import glob
shp = glob.glob(f"{RAW}/taxi_zones/**/*.shp", recursive=True)[0]
contours = gpd.read_file(shp).to_crs(epsg=4326)
contours["LocationID"] = contours["LocationID"].astype(int)
contours[["LocationID", "zone", "borough", "geometry"]].to_file("/home/jovyan/work/data/zones.geojson", driver="GeoJSON")
print(len(contours), "zones enregistrées dans data/zones.geojson")

263 zones enregistrées dans data/zones.geojson


In [8]:
import json
geo = json.load(open("/home/jovyan/work/data/zones.geojson"))
px.choropleth_map(z, geojson=geo, locations="PULocationID", featureidkey="properties.LocationID",
                  color="prix_moyen", hover_name="zone", color_continuous_scale="Viridis",
                  map_style="white-bg", center={"lat": 40.73, "lon": -73.94}, zoom=9.3, opacity=0.75,
                  labels={"prix_moyen": "Prix moyen ($)"},
                  title="Prix moyen d'une course selon la zone de départ")

In [9]:
fig = px.choropleth(z, geojson=geo, locations="PULocationID",
                    featureidkey="properties.LocationID",
                    color="prix_moyen", hover_name="zone",
                    color_continuous_scale="Viridis",
                    labels={"prix_moyen": "Prix moyen ($)"},
                    title="Prix moyen d'une course selon la zone de départ")
fig.update_geos(fitbounds="locations", visible=False)
fig

## 4. De la cellule à l'application

Le dossier `app/` contient une application **Streamlit** qui reprend ces graphiques avec des filtres.
Lancez-la depuis le terminal :

```bash
docker compose --profile app up -d --build
```

puis ouvrez http://localhost:8501. Le code est dans `app/app.py` : modifiez-le, enregistrez, et cliquez sur *Rerun* dans la page.

## Exercices

1. Ajoutez à l'application un filtre sur le **jour de la semaine**.
2. Ajoutez un graphique qui répond à la question : *le pourboire moyen dépend-il de l'heure ?*
3. Remplacez la couleur de la carte par le **nombre de courses**. Pourquoi faut-il alors une échelle logarithmique ?